# Proyecto: Foundation Models en NLP con Structured Outputs
## Generación de Descripciones SEO para Productos de E-commerce

Implementación de OpenAI Structured Outputs con schemas Pydantic para generar y validar
fichas de producto optimizadas para SEO, aplicadas a un catálogo de e-commerce.

## 1. Instalación de dependencias

In [1]:
# !pip install openai pydantic pandas tabulate

## 2. Importación de bibliotecas

In [2]:
import os
import openai
import pandas as pd
from datetime import datetime
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

print("Bibliotecas importadas correctamente.")

Bibliotecas importadas correctamente.


## 3. Configuración de la API Key

> ⚠️ **Nota de seguridad:** API Key ficticia requerida por el enunciado.
> Sustitúyela por la tuya real en local y **nunca** la subas a un repositorio público.

In [3]:
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-2024-08-06"   # snapshot estable con Structured Outputs
print(f"Cliente configurado. Modelo: {MODEL}")

Cliente configurado. Modelo: gpt-4o-2024-08-06


## 4. Definición de Schemas con Pydantic

Clases implementadas:
- **`SearchVolume`** / **`Language`** — Enums de valores permitidos.
- **`SEOKeyword`** — keyword con volumen y relevancia; rangos validados a nivel de campo.
- **`ProductMetadata`** — metadatos tipados (`datetime`, `Language`).
- **`ProductSEODescription`** — schema principal con validaciones de campo granulares
  (`min_length`, `max_length`) y un `model_validator` para reglas cruzadas.

In [4]:
# ── Enums ────────────────────────────────────────────────────────────────────

class SearchVolume(str, Enum):
    alto  = "alto"
    medio = "medio"
    bajo  = "bajo"

class Language(str, Enum):
    es    = "Español (España)"
    es_la = "Español (Latinoamérica)"
    en    = "English"
    fr    = "Français"
    de    = "Deutsch"


# ── SEOKeyword ────────────────────────────────────────────────────────────────

class SEOKeyword(BaseModel):
    """Palabra clave SEO con métricas de búsqueda."""
    keyword: str = Field(
        ...,
        min_length=2, max_length=80,
        description="Palabra clave (2-80 chars)"
    )
    search_volume: SearchVolume = Field(
        ..., description="Volumen: alto | medio | bajo"
    )
    relevance_score: float = Field(
        ..., ge=0.0, le=10.0,
        description="Relevancia 0-10"
    )


# ── ProductMetadata ───────────────────────────────────────────────────────────

class ProductMetadata(BaseModel):
    """Metadatos de trazabilidad de la generación."""
    generated_at:  datetime = Field(..., description="Fecha/hora ISO 8601")
    model_used:    str      = Field(..., min_length=3, max_length=60,
                                    description="Modelo de IA utilizado")
    language:      Language = Field(..., description="Idioma del contenido")
    target_market: str      = Field(..., min_length=2, max_length=100,
                                    description="Mercado objetivo")


# ── ProductSEODescription ─────────────────────────────────────────────────────

class ProductSEODescription(BaseModel):
    """Descripción completa de un producto optimizada para SEO."""

    product_name: str = Field(
        ..., min_length=2, max_length=120,
        description="Nombre comercial del producto"
    )
    product_category: str = Field(
        ..., min_length=2, max_length=80,
        description="Categoría principal"
    )
    short_description: str = Field(
        ..., min_length=50, max_length=160,
        description="Meta description (50-160 chars)"
    )
    long_description: str = Field(
        ..., min_length=300, max_length=3000,
        description="Descripción larga con SEO natural (300-3000 chars)"
    )
    key_features: List[str] = Field(
        ..., min_length=3, max_length=6,
        description="3-6 características principales"
    )
    seo_keywords: List[SEOKeyword] = Field(
        ..., min_length=3, max_length=8,
        description="3-8 palabras clave SEO con métricas"
    )
    seo_score: float = Field(
        ..., ge=0.0, le=100.0,
        description="Puntuación SEO global (0-100)"
    )
    price_range: str = Field(
        ...,
        description="Rango de precio en formato '€min - €max', p.ej. '€89 - €129'"
    )
    metadata: ProductMetadata = Field(..., description="Metadatos de generación")

    # ── Validación granular: cada feature no puede estar vacía ───────────────
    @field_validator("key_features", mode="after")
    @classmethod
    def features_non_empty(cls, v: List[str]) -> List[str]:
        blanks = [i for i, f in enumerate(v) if not f.strip()]
        if blanks:
            raise ValueError(f"key_features vacías en posiciones: {blanks}")
        return v

    # ── Validación granular: price_range debe seguir el formato €X - €Y ─────
    @field_validator("price_range", mode="after")
    @classmethod
    def price_range_format(cls, v: str) -> str:
        import re
        if not re.match(r"^€[\d.,]+ - €[\d.,]+$", v.strip()):
            raise ValueError(
                f"price_range debe tener formato '€min - €max' (recibido: '{v}')"
            )
        return v.strip()

    # ── Reglas cruzadas (model_validator) ────────────────────────────────────
    @model_validator(mode="after")
    def cross_field_rules(self) -> "ProductSEODescription":
        errors: List[str] = []

        # Verificar que short_description no tiene puntos suspensivos de truncado
        if self.short_description.endswith("…") or self.short_description.endswith("..."):
            errors.append("short_description parece truncada (termina en '…' o '...')")

        # Al menos una keyword de volumen alto
        high = [kw for kw in self.seo_keywords if kw.search_volume == SearchVolume.alto]
        if not high:
            errors.append("Debe haber al menos 1 keyword con search_volume='alto'")

        if errors:
            raise ValueError("Reglas cruzadas incumplidas:\n  - " + "\n  - ".join(errors))
        return self


print("✅ Schemas definidos con validaciones granulares por campo:")
print("   SearchVolume | Language | SEOKeyword | ProductMetadata | ProductSEODescription")

✅ Schemas definidos con validaciones granulares por campo:
   SearchVolume | Language | SEOKeyword | ProductMetadata | ProductSEODescription


## 5. Llamada a la API con Structured Outputs

El prompt incluye instrucciones explícitas sobre:
- Formato obligatorio de `price_range` (`€min - €max`)
- Límites exactos de `seo_keywords` y `key_features`
- Longitud de `short_description`

In [5]:
def generate_product_seo(
    product_name: str,
    product_info: str,
    temperature: float = 0.7,
    max_tokens: int = 2000,
) -> ProductSEODescription:
    """
    Genera una descripción SEO completa usando OpenAI Structured Outputs.

    Args:
        product_name:  Nombre del producto.
        product_info:  Características o descripción básica.
        temperature:   Creatividad del modelo (0-1). Default 0.7.
        max_tokens:    Límite de tokens. Default 2000.

    Returns:
        Objeto ProductSEODescription validado por Pydantic.
    """
    system_prompt = (
        "Eres un experto en marketing digital y SEO para e-commerce. "
        "Genera descripciones de productos optimizadas para motores de búsqueda. "
        "Escribe en español, de forma atractiva e integrando palabras clave de forma natural. "
        "Devuelve siempre respuestas estructuradas y completas, respetando estrictamente "
        "los límites de longitud y formato indicados."
    )

    user_prompt = f"""Genera una ficha SEO completa para el siguiente producto:

Producto: {product_name}
Información: {product_info}

Requisitos OBLIGATORIOS (el JSON será validado automáticamente):
- short_description: entre 50 y 160 caracteres exactos, sin puntos suspensivos al final.
- long_description: entre 300 y 3000 caracteres, con keywords integradas de forma natural.
- key_features: EXACTAMENTE entre 3 y 6 elementos; ninguno puede estar vacío.
- seo_keywords: EXACTAMENTE entre 3 y 8 elementos; al menos 1 debe tener search_volume='alto'.
- seo_score: número decimal entre 0.0 y 100.0.
- price_range: USA SIEMPRE el formato '€min - €max' (ejemplo: '€89 - €129'). Sin variaciones.
- generated_at: fecha y hora actual en formato ISO 8601.
- language: usa el valor exacto "Español (España)".
- target_market: mercado hispanohablante más adecuado para el producto."""

    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        response_format=ProductSEODescription,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return completion.choices[0].message.parsed


print("✅ generate_product_seo() definida.")

✅ generate_product_seo() definida.


## 6. Validación y Manejo de Errores

`safe_generate` captura el contenido bruto de la respuesta antes del parseo
para mostrarlo en caso de `ValidationError`, facilitando el debugging.

In [6]:
def safe_generate(
    product_name: str,
    product_info: str,
    **kwargs,
) -> Optional[ProductSEODescription]:
    """Wrapper seguro con logging de contenido bruto en caso de error."""
    raw_content: str = "(sin contenido — la llamada no completó)"
    try:
        # Llamada de bajo nivel para capturar el raw antes del parse
        completion = client.beta.chat.completions.parse(
            model=MODEL,
            messages=[
                {"role": "system", "content": (
                    "Eres experto SEO para e-commerce. Genera fichas de producto en español "
                    "respetando estrictamente los límites de formato indicados."
                )},
                {"role": "user", "content": (
                    f"Producto: {product_name}\nInfo: {product_info}\n"
                    "Formato price_range: '€min - €max'. "
                    "seo_keywords: 3-8 (≥1 alto). key_features: 3-6. "
                    "short_description: 50-160 chars sin puntos suspensivos."
                )},
            ],
            response_format=ProductSEODescription,
            **kwargs,
        )
        # Guardar raw antes del parse por si falla la validación posterior
        raw_content = completion.choices[0].message.content or "(vacío)"
        result = completion.choices[0].message.parsed

        # Segunda validación explícita (detecta errores del model_validator)
        validated = ProductSEODescription.model_validate(result.model_dump(mode="json"))
        print(f"✅ [{product_name}] Generado y validado correctamente.")
        return validated

    except openai.AuthenticationError:
        print("⚠️  API Key inválida. Configura una clave real para llamadas reales.")
        return None
    except openai.APIError as e:
        print(f"❌ Error de API OpenAI para '{product_name}': {e}")
        return None
    except ValidationError as e:
        print(f"❌ ValidationError para '{product_name}' ({len(e.errors())} error/es):")
        for err in e.errors():
            field = " → ".join(str(loc) for loc in err["loc"])
            print(f"   Campo '{field}': {err['msg']}")
        print(f"   ── Contenido bruto recibido (primeros 600 chars) ──")
        print(f"   {raw_content[:600]}")
        return None
    except Exception as e:
        print(f"❌ Error inesperado ({type(e).__name__}) para '{product_name}': {e}")
        print(f"   ── Contenido bruto (primeros 600 chars) ──")
        print(f"   {raw_content[:600]}")
        return None


def seo_indicator(chars: int) -> str:
    """Devuelve un indicador visual del cumplimiento del límite de 160 chars."""
    if chars <= 120:
        return f"✅ {chars}/160 chars (óptimo)"
    elif chars <= 160:
        return f"⚠️  {chars}/160 chars (al límite)"
    else:
        return f"❌ {chars}/160 chars (SUPERA EL LÍMITE)"


def display_product_report(p: dict) -> None:
    """Reporte legible con indicador visual de short_description."""
    sep = "=" * 64
    print(sep)
    print(f"📦 {p['product_name']}  |  {p['product_category']}")
    print(f"💰 {p['price_range']}   📊 SEO Score: {p['seo_score']}/100")
    print("-" * 64)
    sd_len = len(p['short_description'])
    print(f"📝 META DESCRIPTION — {seo_indicator(sd_len)}:")
    print(f"   {p['short_description']}")
    print("-" * 64)
    ld = p['long_description']
    print(f"📄 DESCRIPCIÓN ({len(ld)} chars — preview 200):")
    print(f"   {ld[:200]}{'…' if len(ld) > 200 else ''}")
    print("-" * 64)
    print("✨ CARACTERÍSTICAS:")
    for i, f in enumerate(p['key_features'], 1):
        print(f"   {i}. {f}")
    print("-" * 64)
    print("🔍 PALABRAS CLAVE SEO:")
    for kw in p['seo_keywords']:
        vol = kw['search_volume'] if isinstance(kw['search_volume'], str) else kw['search_volume'].value
        print(f"   • [{vol.upper()}] {kw['keyword']}  (relevancia: {kw['relevance_score']}/10)")
    print("-" * 64)
    m = p['metadata']
    lang = m['language'] if isinstance(m['language'], str) else m['language'].value
    print(f"🤖 {m['model_used']} | {lang} | {m['target_market']}")
    print(sep)


print("✅ safe_generate(), seo_indicator() y display_product_report() definidos.")

✅ safe_generate(), seo_indicator() y display_product_report() definidos.


## 7. Caso de Uso Práctico — Generación SEO para E-commerce

> **Nota:** Estas celdas realizan llamadas reales a la API.
> Sustituye `sk-YOUR_FAKE_API_KEY` por tu clave real antes de ejecutarlas.

In [7]:
productos_input = [
    {
        "product_name": "SmartFit Pro X500",
        "product_info": (
            "Reloj inteligente deportivo con pantalla AMOLED 1.8\", GPS integrado, "
            "monitor cardíaco 24/7, SpO2, resistencia 5ATM, 14 días de batería, "
            "compatible iOS y Android, +100 modos deportivos, notificaciones y control de música."
        ),
    },
    {
        "product_name": "LimpiaCasa Premium",
        "product_info": (
            "Servicio profesional de limpieza del hogar a domicilio. Personal verificado, "
            "productos eco-friendly, modalidades básica/profunda/postconstrucción, "
            "disponible 7 días, seguro de responsabilidad civil, app para reservas."
        ),
    },
    {
        "product_name": "SoundElite ANC 900",
        "product_info": (
            "Auriculares inalámbricos over-ear con cancelación activa de ruido, drivers 40mm, "
            "35h de autonomía con ANC, carga rápida 10min=3h, Bluetooth 5.3 multiconexión, "
            "micrófono con IA, Hi-Res Audio, plegables, funda incluida."
        ),
    },
]

resultados: List[ProductSEODescription] = []

for prod in productos_input:
    r = safe_generate(**prod)
    if r:
        resultados.append(r)
        display_product_report(r.model_dump())
    print()

print(f"📊 Productos generados con éxito: {len(resultados)}/{len(productos_input)}")

❌ Error de API OpenAI para 'SmartFit Pro X500': Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

❌ Error de API OpenAI para 'LimpiaCasa Premium': Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

❌ Error de API OpenAI para 'SoundElite ANC 900': Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-error

## 8. Simulación de Respuesta (sin API Key real)

Validación completa del pipeline Pydantic con datos de ejemplo.

In [8]:
mock_responses = [
    {
        "product_name": "SmartFit Pro X500", "product_category": "Electrónica / Wearables",
        "short_description": "Reloj deportivo GPS, 14 días batería y +100 modos. Monitoriza tu salud 24/7.",
        "long_description": (
            "El SmartFit Pro X500 es el compañero perfecto para deportistas que buscan un reloj "
            "inteligente completo a precio competitivo. Con GPS integrado de alta precisión registra "
            "rutas de running, ciclismo o senderismo sin necesitar el móvil. Su pantalla AMOLED de "
            "1.8 pulgadas ofrece visibilidad excepcional bajo la luz solar directa. El monitor "
            "cardíaco continuo y el sensor SpO2 analizan tu rendimiento en tiempo real, clave para "
            "entrenamientos de alta intensidad. Compatible con iOS y Android, gestiona notificaciones "
            "y controla tu música sin interrumpir el entrenamiento. Resistente al agua 5ATM. "
            "Con 14 días de autonomía es el smartwatch con mejor batería de su categoría."
        ),
        "key_features": ["GPS integrado sin el móvil", "AMOLED 1.8\" adaptativo",
                         "Monitor cardíaco 24/7 + SpO2", "14 días de batería", "5ATM natación", "+100 modos"],
        "seo_keywords": [
            {"keyword": "reloj inteligente deportivo",     "search_volume": "alto",  "relevance_score": 9.5},
            {"keyword": "smartwatch GPS integrado",        "search_volume": "alto",  "relevance_score": 9.2},
            {"keyword": "reloj smartwatch mejor batería",  "search_volume": "medio", "relevance_score": 8.7},
            {"keyword": "smartwatch impermeable running",  "search_volume": "medio", "relevance_score": 8.4},
            {"keyword": "reloj fitness SpO2",              "search_volume": "bajo",  "relevance_score": 7.6},
        ],
        "seo_score": 87.5, "price_range": "€89 - €129",
        "metadata": {"generated_at": datetime.now().isoformat(), "model_used": "gpt-4o-2024-08-06",
                     "language": "Español (España)", "target_market": "España y Latinoamérica"},
    },
    {
        "product_name": "LimpiaCasa Premium", "product_category": "Servicios del Hogar",
        "short_description": "Limpieza profesional a domicilio con personal verificado y productos eco. Reserva en minutos.",
        "long_description": (
            "LimpiaCasa Premium ofrece un servicio profesional de limpieza del hogar a domicilio "
            "con personal altamente cualificado y verificado. Utilizamos productos eco-friendly "
            "seguros para niños y mascotas. Tres modalidades: limpieza básica, profunda y "
            "postconstrucción. Operamos los 7 días de la semana con total flexibilidad horaria. "
            "Incluimos seguro de responsabilidad civil en todas las limpiezas. Nuestra app permite "
            "reservar, pagar y hacer seguimiento en tiempo real del servicio de limpieza del hogar. "
            "El servicio de limpieza profesional más valorado de España con más de 10.000 hogares "
            "limpios cada mes. Garantía de satisfacción o repetimos la limpieza sin coste adicional."
        ),
        "key_features": ["Personal verificado y cualificado", "Productos eco-friendly",
                         "3 modalidades de servicio", "Disponible 7 días", "Seguro RC incluido", "App de reservas"],
        "seo_keywords": [
            {"keyword": "servicio limpieza hogar domicilio", "search_volume": "alto",  "relevance_score": 9.3},
            {"keyword": "empresa limpieza profesional",      "search_volume": "alto",  "relevance_score": 9.0},
            {"keyword": "limpieza ecológica domicilio",      "search_volume": "medio", "relevance_score": 8.2},
            {"keyword": "servicio limpieza postconstruccion","search_volume": "bajo",  "relevance_score": 7.8},
        ],
        "seo_score": 83.0, "price_range": "€45 - €120",
        "metadata": {"generated_at": datetime.now().isoformat(), "model_used": "gpt-4o-2024-08-06",
                     "language": "Español (España)", "target_market": "España"},
    },
    {
        "product_name": "SoundElite ANC 900", "product_category": "Electrónica / Audio",
        "short_description": "Auriculares ANC con 35h de batería, Hi-Res Audio y carga rápida. Silencio total garantizado.",
        "long_description": (
            "Los SoundElite ANC 900 redefinen la experiencia de audio con cancelación activa de "
            "ruido que elimina hasta el 98% del sonido ambiental. Sus drivers de 40mm con "
            "certificación Hi-Res Audio reproducen cada detalle con claridad excepcional. "
            "Con 35 horas de autonomía y carga rápida (10 minutos = 3 horas) nunca te quedarás "
            "sin música. Bluetooth 5.3 con conexión simultánea a dos dispositivos. Micrófono con "
            "inteligencia artificial que filtra el ruido en llamadas. Diseño plegable con funda "
            "rígida para viaje. Los mejores auriculares inalámbricos con cancelación de ruido "
            "del mercado, ideales para trabajo, viajes y estudio. Disponibles en negro, blanco y azul."
        ),
        "key_features": ["ANC elimina 98% ruido ambiental", "Hi-Res Audio certificado",
                         "35h autonomía + carga rápida 10min", "Bluetooth 5.3 multiconexión",
                         "Micrófono IA para llamadas", "Diseño plegable con funda"],
        "seo_keywords": [
            {"keyword": "auriculares cancelación ruido",     "search_volume": "alto",  "relevance_score": 9.4},
            {"keyword": "auriculares inalámbricos ANC",      "search_volume": "alto",  "relevance_score": 9.1},
            {"keyword": "auriculares Hi-Res Audio",          "search_volume": "medio", "relevance_score": 8.5},
            {"keyword": "auriculares bluetooth larga duracion","search_volume": "medio","relevance_score": 8.0},
            {"keyword": "mejores auriculares trabajo oficina","search_volume": "bajo",  "relevance_score": 7.4},
        ],
        "seo_score": 91.0, "price_range": "€149 - €199",
        "metadata": {"generated_at": datetime.now().isoformat(), "model_used": "gpt-4o-2024-08-06",
                     "language": "Español (España)", "target_market": "España y Latinoamérica"},
    },
]

print("🔄 Validando respuestas simuladas contra el schema Pydantic...\n")
productos_validados: List[ProductSEODescription] = []

for raw in mock_responses:
    try:
        p = ProductSEODescription.model_validate(raw)
        productos_validados.append(p)
        print(f"✅ '{p.product_name}' — validación OK")
        display_product_report(p.model_dump())
    except ValidationError as e:
        print(f"❌ Error en '{raw.get('product_name','?')}': {e}")
    print()

print(f"\n📊 {len(productos_validados)}/{len(mock_responses)} productos validados correctamente.")

🔄 Validando respuestas simuladas contra el schema Pydantic...

✅ 'SmartFit Pro X500' — validación OK
📦 SmartFit Pro X500  |  Electrónica / Wearables
💰 €89 - €129   📊 SEO Score: 87.5/100
----------------------------------------------------------------
📝 META DESCRIPTION — ✅ 76/160 chars (óptimo):
   Reloj deportivo GPS, 14 días batería y +100 modos. Monitoriza tu salud 24/7.
----------------------------------------------------------------
📄 DESCRIPCIÓN (670 chars — preview 200):
   El SmartFit Pro X500 es el compañero perfecto para deportistas que buscan un reloj inteligente completo a precio competitivo. Con GPS integrado de alta precisión registra rutas de running, ciclismo o …
----------------------------------------------------------------
✨ CARACTERÍSTICAS:
   1. GPS integrado sin el móvil
   2. AMOLED 1.8" adaptativo
   3. Monitor cardíaco 24/7 + SpO2
   4. 14 días de batería
   5. 5ATM natación
   6. +100 modos
----------------------------------------------------------------
🔍 PA

## 9. Test de Validación con Datos Inválidos

In [9]:
casos_invalidos = [
    {
        "nombre": "price_range con formato incorrecto",
        "data": {
            "product_name": "Test A", "product_category": "Test",
            "short_description": "Descripción de prueba con al menos cincuenta caracteres para pasar.",
            "long_description": "A" * 300,
            "key_features": ["F1", "F2", "F3"],
            "seo_keywords": [
                {"keyword": "test kw", "search_volume": "alto", "relevance_score": 8.0},
                {"keyword": "otro kw", "search_volume": "medio", "relevance_score": 7.0},
                {"keyword": "tercer kw","search_volume": "bajo", "relevance_score": 6.0},
            ],
            "seo_score": 75.0,
            "price_range": "89 euros - 129 euros",   # ❌ formato incorrecto
            "metadata": {"generated_at": datetime.now().isoformat(), "model_used": "gpt-4o-2024-08-06",
                         "language": "Español (España)", "target_market": "España"},
        },
    },
    {
        "nombre": "relevance_score > 10 y search_volume inválido",
        "data": {
            "product_name": "Test B", "product_category": "Test",
            "short_description": "Descripción de prueba con al menos cincuenta caracteres para pasar.",
            "long_description": "B" * 300,
            "key_features": ["F1", "F2", "F3"],
            "seo_keywords": [
                {"keyword": "kw1", "search_volume": "ultra", "relevance_score": 15.0},  # ❌ ambos
            ],   # ❌ sólo 1 keyword (mínimo 3)
            "seo_score": 75.0, "price_range": "€89 - €129",
            "metadata": {"generated_at": datetime.now().isoformat(), "model_used": "gpt-4o-2024-08-06",
                         "language": "Español (España)", "target_market": "España"},
        },
    },
    {
        "nombre": "short_description supera 160 chars",
        "data": {
            "product_name": "Test C", "product_category": "Test",
            "short_description": "X" * 161,   # ❌ > 160
            "long_description": "C" * 300,
            "key_features": ["F1", "F2", "F3"],
            "seo_keywords": [
                {"keyword": "kw1", "search_volume": "alto",  "relevance_score": 8.0},
                {"keyword": "kw2", "search_volume": "medio", "relevance_score": 7.0},
                {"keyword": "kw3", "search_volume": "bajo",  "relevance_score": 6.0},
            ],
            "seo_score": 75.0, "price_range": "€89 - €129",
            "metadata": {"generated_at": datetime.now().isoformat(), "model_used": "gpt-4o-2024-08-06",
                         "language": "Español (España)", "target_market": "España"},
        },
    },
]

print("🧪 Test de validaciones granulares por campo:\n")
for caso in casos_invalidos:
    print(f"  Caso: {caso['nombre']}")
    try:
        ProductSEODescription.model_validate(caso["data"])
        print("  ⚠️  No se detectaron errores (inesperado)\n")
    except ValidationError as e:
        print(f"  ✅ {len(e.errors())} error(es) detectado(s) correctamente:")
        for err in e.errors():
            field = " → ".join(str(loc) for loc in err["loc"])
            print(f"     ❌ Campo '{field}': {err['msg']}")
        print()

🧪 Test de validaciones granulares por campo:

  Caso: price_range con formato incorrecto
  ✅ 1 error(es) detectado(s) correctamente:
     ❌ Campo 'price_range': Value error, price_range debe tener formato '€min - €max' (recibido: '89 euros - 129 euros')

  Caso: relevance_score > 10 y search_volume inválido
  ✅ 2 error(es) detectado(s) correctamente:
     ❌ Campo 'seo_keywords → 0 → search_volume': Input should be 'alto', 'medio' or 'bajo'
     ❌ Campo 'seo_keywords → 0 → relevance_score': Input should be less than or equal to 10

  Caso: short_description supera 160 chars
  ✅ 1 error(es) detectado(s) correctamente:
     ❌ Campo 'short_description': String should have at most 160 characters



## 10. Catálogo y Análisis Cuantitativo — Integración en E-commerce

Los productos validados se consolidan en un DataFrame de pandas que simula la tabla `products`
de un CMS. Se incluye un análisis cuantitativo: comparación de `seo_score`, longitud de
`short_description` y filtrado por rango de precio.

In [10]:
import re

def parse_price(price_range: str) -> tuple:
    """Extrae (price_min, price_max) numéricos de un string '€X - €Y'."""
    nums = re.findall(r"[\d.,]+", price_range)
    if len(nums) >= 2:
        return float(nums[0].replace(",", ".")), float(nums[1].replace(",", "."))
    return (0.0, 0.0)

def to_db_row(p: ProductSEODescription) -> dict:
    """
    Mapea ProductSEODescription → columnas de la tabla `products` de un CMS.
    Campos: sku, name, category, meta_description, body_html,
            tags, seo_score, meta_ok, price_min, price_max, generated_at, ai_model.
    """
    price_min, price_max = parse_price(p.price_range)
    sd_len = len(p.short_description)
    return {
        "sku":              p.product_name.lower().replace(" ", "-"),
        "name":             p.product_name,
        "category":         p.product_category,
        "meta_description": p.short_description,
        "meta_chars":       sd_len,
        "meta_ok":          "✅" if sd_len <= 160 else "❌",
        "body_html":        p.long_description[:120] + "…",
        "tags":             ", ".join(kw.keyword for kw in p.seo_keywords),
        "seo_score":        p.seo_score,
        "price_min":        price_min,
        "price_max":        price_max,
        "generated_at":     p.metadata.generated_at.isoformat(),
        "ai_model":         p.metadata.model_used,
    }

df = pd.DataFrame([to_db_row(p) for p in productos_validados])

# ── Vista general ─────────────────────────────────────────────────────────
print("=" * 64)
print("📋 TABLA products — vista del catálogo generado")
print("=" * 64)
cols = ["sku", "category", "seo_score", "meta_chars", "meta_ok", "price_min", "price_max"]
print(df[cols].to_string(index=False))

# ── Análisis cuantitativo: ranking SEO ────────────────────────────────────
print("\n" + "=" * 64)
print("📊 RANKING por SEO Score (mayor a menor)")
print("=" * 64)
ranking = df[["name", "seo_score", "meta_chars", "meta_ok"]].sort_values("seo_score", ascending=False)
for _, row in ranking.iterrows():
    bar = "█" * int(row["seo_score"] / 5)
    print(f"  {row['seo_score']:5.1f}/100  {bar:<20}  {row['name']}  |  meta: {row['meta_ok']} {row['meta_chars']} chars")

# ── Análisis cuantitativo: filtro por precio ──────────────────────────────
print("\n" + "=" * 64)
print("💰 FILTRO: productos con precio máximo ≤ €150")
print("=" * 64)
baratos = df[df["price_max"] <= 150][["name", "price_min", "price_max", "seo_score"]]
if baratos.empty:
    print("  (ningún producto cumple el filtro)")
else:
    print(baratos.to_string(index=False))

# ── Estadísticas globales ─────────────────────────────────────────────────
print("\n" + "=" * 64)
print("📈 ESTADÍSTICAS GLOBALES DEL CATÁLOGO")
print("=" * 64)
print(f"  Productos en catálogo:   {len(df)}")
print(f"  SEO score medio:         {df['seo_score'].mean():.1f}/100")
print(f"  SEO score máximo:        {df['seo_score'].max():.1f}  ({df.loc[df['seo_score'].idxmax(), 'name']})")
print(f"  SEO score mínimo:        {df['seo_score'].min():.1f}  ({df.loc[df['seo_score'].idxmin(), 'name']})")
print(f"  Meta descriptions OK:    {(df['meta_ok'] == '✅').sum()}/{len(df)}")
print(f"  Precio medio (max):      €{df['price_max'].mean():.0f}")
print(f"  Rango precios (min-max): €{df['price_min'].min():.0f} — €{df['price_max'].max():.0f}")
print("=" * 64)

📋 TABLA products — vista del catálogo generado
               sku                category  seo_score  meta_chars meta_ok  price_min  price_max
 smartfit-pro-x500 Electrónica / Wearables       87.5          76       ✅       89.0      129.0
limpiacasa-premium     Servicios del Hogar       83.0          93       ✅       45.0      120.0
soundelite-anc-900     Electrónica / Audio       91.0          92       ✅      149.0      199.0

📊 RANKING por SEO Score (mayor a menor)
   91.0/100  ██████████████████    SoundElite ANC 900  |  meta: ✅ 92 chars
   87.5/100  █████████████████     SmartFit Pro X500  |  meta: ✅ 76 chars
   83.0/100  ████████████████      LimpiaCasa Premium  |  meta: ✅ 93 chars

💰 FILTRO: productos con precio máximo ≤ €150
              name  price_min  price_max  seo_score
 SmartFit Pro X500       89.0      129.0       87.5
LimpiaCasa Premium       45.0      120.0       83.0

📈 ESTADÍSTICAS GLOBALES DEL CATÁLOGO
  Productos en catálogo:   3
  SEO score medio:         87.2/100

## 11. Resumen del Proyecto

In [11]:
print("=" * 64)
print("RESUMEN DEL PROYECTO")
print("=" * 64)
items = {
    "Schemas Pydantic": [
        "SearchVolume, Language (Enum)",
        "SEOKeyword (field validators: min/max_length, ge/le)",
        "ProductMetadata (datetime tipado, Language Enum)",
        "ProductSEODescription (field_validator price_range y key_features,",
        "  model_validator para reglas cruzadas)",
    ],
    "Modelo OpenAI": ["gpt-4o-2024-08-06 — Structured Outputs estable"],
    "Método API": ["client.beta.chat.completions.parse(response_format=ProductSEODescription)"],
    "Prompt": ["Formato price_range explícito ('€min - €max')",
               "Límites exactos de keywords y features indicados",
               "Restricción de longitud de short_description"],
    "Validaciones": ["Granulares por campo (min/max_length, ge/le, Enum, datetime)",
                     "field_validator: price_range regex, key_features no vacías",
                     "model_validator: reglas cruzadas (keyword alto, truncado)"],
    "Manejo de errores": ["safe_generate() registra raw_content antes del parse",
                          "ValidationError muestra campo exacto + contenido bruto"],
    "Análisis cuantitativo": ["Ranking por seo_score con barra visual",
                              "Filtro por rango de precio",
                              "Estadísticas globales del catálogo"],
    "Integración e-commerce": ["DataFrame → tabla `products` de CMS",
                               "Columnas: sku, meta_description, tags, body_html, price_min/max…",
                               "Indicador visual meta_ok (✅/❌) por producto"],
}
for section, lines in items.items():
    print(f"\n  {section}:")
    for line in lines:
        print(f"    • {line}")
print("\n" + "=" * 64)

RESUMEN DEL PROYECTO

  Schemas Pydantic:
    • SearchVolume, Language (Enum)
    • SEOKeyword (field validators: min/max_length, ge/le)
    • ProductMetadata (datetime tipado, Language Enum)
    • ProductSEODescription (field_validator price_range y key_features,
    •   model_validator para reglas cruzadas)

  Modelo OpenAI:
    • gpt-4o-2024-08-06 — Structured Outputs estable

  Método API:
    • client.beta.chat.completions.parse(response_format=ProductSEODescription)

  Prompt:
    • Formato price_range explícito ('€min - €max')
    • Límites exactos de keywords y features indicados
    • Restricción de longitud de short_description

  Validaciones:
    • Granulares por campo (min/max_length, ge/le, Enum, datetime)
    • field_validator: price_range regex, key_features no vacías
    • model_validator: reglas cruzadas (keyword alto, truncado)

  Manejo de errores:
    • safe_generate() registra raw_content antes del parse
    • ValidationError muestra campo exacto + contenido bruto